In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!mkdir -p /content/drive/MyDrive/WeedProject/zips
!mkdir -p /content/drive/MyDrive/WeedProject/models


In [ ]:
!mv /content/drive/MyDrive/WeedProject/PaimioSB2023_*.zip \
/content/drive/MyDrive/WeedProject/zips/


mv: cannot stat '/content/drive/MyDrive/WeedProject/PaimioSB2023_*.zip': No such file or directory


In [ ]:
!mkdir -p /content/dataset/images
!mkdir -p /content/dataset/labels


In [ ]:
import zipfile
import os

def unzip(src, dest):
    with zipfile.ZipFile(src, 'r') as z:
        z.extractall(dest)

base = "/content/drive/MyDrive/WeedProject/zips"

unzip(f"{base}/PaimioSB2023_images_part1.zip", "/content/dataset/images")
unzip(f"{base}/PaimioSB2023_images_part2.zip", "/content/dataset/images")
unzip(f"{base}/PaimioSB2023_images_part3.zip", "/content/dataset/images")
unzip(f"{base}/PaimioSB2023_labels.zip", "/content/dataset/labels")

print("✅ Unzipped")


✅ Unzipped


In [ ]:
!ls /content/dataset/images


PaimioSB2023_images_part1  PaimioSB2023_images_part2  PaimioSB2023_images_part3


In [ ]:
import os
import shutil

def flatten_folder(root_dir):
    for folder in os.listdir(root_dir):
        folder_path = os.path.join(root_dir, folder)

        if os.path.isdir(folder_path):
            for file in os.listdir(folder_path):
                shutil.move(
                    os.path.join(folder_path, file),
                    root_dir
                )
            os.rmdir(folder_path)

    print(f"✅ Flattened {root_dir}")

# Flatten images
flatten_folder("/content/dataset/images")


✅ Flattened /content/dataset/images


In [ ]:
!ls /content/dataset/images | head


G0022035_b_0_0.jpg
G0022035_b_0_1.jpg
G0022035_b_0_2.jpg
G0022035_b_0_3.jpg
G0022035_b_0_4.jpg
G0022035_b_1_0.jpg
G0022035_b_1_1.jpg
G0022035_b_1_2.jpg
G0022035_b_1_3.jpg
G0022035_b_1_4.jpg


In [ ]:
!ls /content/dataset/labels


PaimioSB2023_labels


In [ ]:
flatten_folder("/content/dataset/labels")


✅ Flattened /content/dataset/labels


In [ ]:
!ls /content/dataset/labels | head

G0022035_b_0_0.txt
G0022035_b_0_1.txt
G0022035_b_0_2.txt
G0022035_b_0_3.txt
G0022035_b_0_4.txt
G0022035_b_1_0.txt
G0022035_b_1_1.txt
G0022035_b_1_2.txt
G0022035_b_1_3.txt
G0022035_b_1_4.txt


In [ ]:
!find /content/dataset/labels -type f -iname "*.txt" | wc -l


18594


In [ ]:
import os
import random
import shutil

# Fix seed for reproducibility
random.seed(42)

images_dir = "/content/dataset/images"
labels_dir = "/content/dataset/labels"

images = [f for f in os.listdir(images_dir) if f.endswith((".jpg", ".png"))]
random.shuffle(images)

split_index = int(0.8 * len(images))

train_imgs = images[:split_index]
val_imgs = images[split_index:]

# Create folders
for folder in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(f"/content/dataset/{folder}", exist_ok=True)

# Move files
for img in train_imgs:
    shutil.move(os.path.join(images_dir, img),
                "/content/dataset/images/train")
    shutil.move(os.path.join(labels_dir, img.rsplit(".",1)[0] + ".txt"),
                "/content/dataset/labels/train")

for img in val_imgs:
    shutil.move(os.path.join(images_dir, img),
                "/content/dataset/images/val")
    shutil.move(os.path.join(labels_dir, img.rsplit(".",1)[0] + ".txt"),
                "/content/dataset/labels/val")

print("✅ Train/Val split completed")


✅ Train/Val split completed


In [ ]:
yaml_content = """
train: /content/dataset/images/train
val: /content/dataset/images/val

nc: 2
names:
  - sugarbeet
  - weed
"""

with open("/content/dataset/sugarbeet.yaml", "w") as f:
    f.write(yaml_content)

print("✅ YAML file created")


✅ YAML file created


In [ ]:
!zip -r dataset_ready.zip /content/dataset


Streaming output truncated to the last 5000 lines.
  adding: content/dataset/labels/train/G0022331_3_1.txt (deflated 55%)
  adding: content/dataset/labels/train/P1143_1_0.txt (deflated 66%)
  adding: content/dataset/labels/train/G0022689_1_0.txt (deflated 63%)
  adding: content/dataset/labels/train/G0032892_3_4.txt (deflated 67%)
  adding: content/dataset/labels/train/G0022126_1_0.txt (deflated 63%)
  adding: content/dataset/labels/train/P1133_0_2.txt (deflated 68%)
  adding: content/dataset/labels/train/G0032970_3_1.txt (deflated 55%)
  adding: content/dataset/labels/train/G0022536_2_4.txt (deflated 63%)
  adding: content/dataset/labels/train/G0022055_2_3.txt (deflated 64%)
  adding: content/dataset/labels/train/G0022291_2_1.txt (deflated 61%)
  adding: content/dataset/labels/train/G0022504_1_1.txt (deflated 65%)
  adding: content/dataset/labels/train/G0022532_2_4.txt (deflated 61%)
  adding: content/dataset/labels/train/G0022414_1_0.txt (deflated 64%)
  adding: content/dataset/labels

In [ ]:
!ls /content/dataset/images


train  val


In [ ]:
!ls /content/dataset/labels


labels.txt  train  val


In [ ]:
!find /content/dataset/images -type f | wc -l


18593


In [ ]:
!find /content/dataset/labels -type f | wc -l


18594


In [ ]:
!rm /content/dataset/labels/labels.txt


In [ ]:
import os

image_train = set(os.listdir("/content/dataset/images/train"))
image_val = set(os.listdir("/content/dataset/images/val"))
images = image_train.union(image_val)

label_train = set([f.replace(".txt",".jpg") for f in os.listdir("/content/dataset/labels/train")])
label_val = set([f.replace(".txt",".jpg") for f in os.listdir("/content/dataset/labels/val")])
labels = label_train.union(label_val)

missing_images = labels - images
missing_labels = images - labels

print("Missing image for label:", missing_images)
print("Missing label for image:", missing_labels)


Missing image for label: set()
Missing label for image: set()


In [ ]:
!find /content/dataset/images -type f | wc -l

18593


In [ ]:
!find /content/dataset/labels -type f | wc -l

18593


In [ ]:
!zip -r dataset_ready.zip /content/dataset


Streaming output truncated to the last 5000 lines.
updating: content/dataset/labels/train/G0022331_3_1.txt (deflated 55%)
updating: content/dataset/labels/train/P1143_1_0.txt (deflated 66%)
updating: content/dataset/labels/train/G0022689_1_0.txt (deflated 63%)
updating: content/dataset/labels/train/G0032892_3_4.txt (deflated 67%)
updating: content/dataset/labels/train/G0022126_1_0.txt (deflated 63%)
updating: content/dataset/labels/train/P1133_0_2.txt (deflated 68%)
updating: content/dataset/labels/train/G0032970_3_1.txt (deflated 55%)
updating: content/dataset/labels/train/G0022536_2_4.txt (deflated 63%)
updating: content/dataset/labels/train/G0022055_2_3.txt (deflated 64%)
updating: content/dataset/labels/train/G0022291_2_1.txt (deflated 61%)
updating: content/dataset/labels/train/G0022504_1_1.txt (deflated 65%)
updating: content/dataset/labels/train/G0022532_2_4.txt (deflated 61%)
updating: content/dataset/labels/train/G0022414_1_0.txt (deflated 64%)
updating: content/dataset/labels

In [ ]:
!ls /content/dataset/images


train  val


In [ ]:
!ls /content/dataset/labels


train  val


In [ ]:
!find /content/dataset/images -type f | wc -l

18593


In [ ]:
!find /content/dataset/labels -type f | wc -l

18593


In [ ]:
!mv -f dataset_ready.zip /content/drive/MyDrive/WeedProject/

In [ ]:
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.5 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
!head -n 5 /content/dataset/labels/train/*.txt


Streaming output truncated to the last 5000 lines.

==> /content/dataset/labels/train/P1699_1_2.txt <==
0 0.715625 0.1546875 0.1375 0.15625
0 0.52109375 0.3921875 0.1234375 0.128125
0 0.8875 0.36875 0.0875 0.10625
0 0.66796875 0.6828125 0.1015625 0.096875
0 0.60859375 0.3296875 0.1140625 0.146875

==> /content/dataset/labels/train/P1699_2_0.txt <==
1 0.3875 0.35234375 0.0625 0.0671875
1 0.10625 0.32421875 0.0875 0.0828125
1 0.1171875 0.128125 0.06875 0.04375
1 0.609375 0.77734375 0.053125 0.0484375
1 0.81015625 0.0875 0.0421875 0.04375

==> /content/dataset/labels/train/P1699_2_1.txt <==
0 0.79765625 0.30625 0.0546875 0.084375
0 0.71796875 0.89453125 0.0515625 0.0578125
0 0.8 0.60390625 0.05625 0.0546875
0 0.596875 0.8765625 0.05625 0.046875
1 0.16171875 0.18671875 0.0515625 0.0484375

==> /content/dataset/labels/train/P1700_0_1.txt <==
0 0.1234375 0.25703125 0.1375 0.1546875
0 0.62890625 0.69453125 0.0859375 0.1140625
0 0.49375 0.31796875 0.10625 0.1296875
0 0.88828125 0.81875 0.09843

In [ ]:
yaml_content = """
path: /content/dataset

train: images/train
val: images/val

nc: 2
names:
  - sugarbeet
  - weed
"""

with open("/content/dataset/sugarbeet.yaml", "w") as f:
    f.write(yaml_content)

print("✅ YAML created successfully")


✅ YAML created successfully


In [ ]:
model = YOLO('yolov8n.pt')

model.train(
    data='/content/dataset/sugarbeet.yaml',
    epochs=2,
    imgsz=640,
    batch=16,
    project='/content/drive/MyDrive/WeedProject/weed_training',
    name='exp1',
    exist_ok=True
)


NameError: name 'YOLO' is not defined

In [ ]:
!nvidia-smi


Mon Mar  2 05:30:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!cp /content/drive/MyDrive/WeedProject/dataset_ready.zip /content/
!unzip -q dataset_ready.zip


replace content/dataset/images/val/G0032895_e_2_0.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
A
A
A


In [ ]:
!rm -rf /content/dataset


In [ ]:
!unzip -oq dataset_ready.zip


In [ ]:
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.4 MB/s eta 0:00:00


In [ ]:
yaml_content = """
path: /content/dataset

train: images/train
val: images/val

nc: 2
names:
  - sugarbeet
  - weed
"""

with open("/content/dataset/sugarbeet.yaml", "w") as f:
    f.write(yaml_content)

print("YAML recreated")


FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset/sugarbeet.yaml'

In [ ]:
!ls /content/content


dataset


In [ ]:
!mv /content/content/dataset /content/
!rm -rf /content/content

In [ ]:
!ls /content
!ls /content/content


dataset  dataset_ready.zip  drive  sample_data
ls: cannot access '/content/content': No such file or directory


In [ ]:
!ls /content/dataset


images	labels	sugarbeet.yaml


In [ ]:
!ls /content/drive/MyDrive/WeedProject/weed_training/exp1/weights


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

model.train(
    data='/content/dataset/sugarbeet.yaml',
    epochs=2,
    imgsz=640,
    batch=16,
    project='/content/drive/MyDrive/WeedProject/weed_training',
    name='exp1',
    exist_ok=True
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/sugarbeet.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=2, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, in

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79a0091c31a0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
!ls /content/drive/MyDrive/WeedProject/weed_training/exp1


args.yaml			 labels.jpg		val_batch0_pred.jpg
BoxF1_curve.png			 results.csv		val_batch1_labels.jpg
BoxP_curve.png			 results.png		val_batch1_pred.jpg
BoxPR_curve.png			 train_batch0.jpg	val_batch2_labels.jpg
BoxR_curve.png			 train_batch1.jpg	val_batch2_pred.jpg
confusion_matrix_normalized.png  train_batch2.jpg	weights
confusion_matrix.png		 val_batch0_labels.jpg


In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/WeedProject/weed_training/exp1/weights/last.pt')

model.train(
    data='/content/dataset/sugarbeet.yaml',
    epochs=4,   # total epochs you now want
    imgsz=640,
    batch=16,
    project='/content/drive/MyDrive/WeedProject/weed_training',
    name='exp1',
    exist_ok=True
)


Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/sugarbeet.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=4, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/WeedProject/weed_training/exp1/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp1, nbs=64, nms=False, opset=None, optimize=False, op

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79a1066964b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
import pandas as pd

# Path to your current results.csv
csv_path = "/content/drive/MyDrive/WeedProject/weed_training/exp1/results.csv"

# Load your current CSV
df = pd.read_csv(csv_path)

# First 2 epoch data (from your old logs)
first2 = [
    {
        "epoch": 1, "time": 0, "train/box_loss": 1.294, "train/cls_loss": 1.242, "train/dfl_loss": 1.077,
        "metrics/precision(B)": 0.793, "metrics/recall(B)": 0.765, "metrics/mAP50(B)": 0.845, "metrics/mAP50-95(B)": 0.599,
        "val/box_loss": 0.99069, "val/cls_loss": 0.74981, "val/dfl_loss": 0.99218,
        "lr/pg0": 0.000555069, "lr/pg1": 0.000555069, "lr/pg2": 0.000555069
    },
    {
        "epoch": 2, "time": 0, "train/box_loss": 1.079, "train/cls_loss": 0.8049, "train/dfl_loss": 1.001,
        "metrics/precision(B)": 0.836, "metrics/recall(B)": 0.817, "metrics/mAP50(B)": 0.893, "metrics/mAP50-95(B)": 0.654,
        "val/box_loss": 0.91976, "val/cls_loss": 0.63349, "val/dfl_loss": 0.97284,
        "lr/pg0": 0.000835829, "lr/pg1": 0.000835829, "lr/pg2": 0.000835829
    }
]

df_first2 = pd.DataFrame(first2)

# Shift current epochs by +2
df['epoch'] = df['epoch'] + 2

# Combine
df_full = pd.concat([df_first2, df], ignore_index=True)

# Save back
df_full.to_csv(csv_path, index=False)
print("✅ Full 6-epoch results.csv created and saved!")


✅ Full 6-epoch results.csv created and saved!


In [ ]:
from ultralytics import YOLO

# Load your trained model
model = YOLO("/content/drive/MyDrive/WeedProject/weed_training/exp1/weights/last.pt")


In [ ]:
# This will regenerate all standard plots from results.csv
model.plot_results("/content/drive/MyDrive/WeedProject/weed_training/exp1/results.csv")


AttributeError: 'DetectionModel' object has no attribute 'plot_results'

In [ ]:
from ultralytics.utils.plotting import plot_results


In [ ]:
csv_path = "/content/drive/MyDrive/WeedProject/weed_training/exp1/results.csv"

# This will overwrite results.png with all 6 epochs
plot_results(csv_path)


In [ ]:
model = YOLO("/content/drive/MyDrive/WeedProject/weed_training/exp1/weights/best.pt")
model.val(save_conf=True, plots=True)


Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2852.6±641.8 MB/s, size: 342.2 KB)
val: Scanning /content/dataset/labels/val.cache... 3719 images, 47 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3719/3719 866.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 233/233 4.0it/s 58.9s
                   all       3719      67235      0.868      0.837      0.917        0.7
             sugarbeet       3580      36920      0.901      0.943      0.973       0.79
                  weed       3424      30315      0.836       0.73       0.86      0.611
Speed: 1.9ms preprocess, 4.0ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to /content/runs/detect/val


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79a15c0bfd10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.9 MB/s eta 0:00:00


In [ ]:
!cp /content/drive/MyDrive/WeedProject/dataset_ready.zip /content/
!unzip -q dataset_ready.zip


In [ ]:
!ls /content/


content  dataset_ready.zip  drive  sample_data


In [ ]:
!unzip -q dataset_ready.zip


replace content/dataset/images/val/G0032895_e_2_0.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A

A


In [ ]:
!rm -rf /content/dataset
!unzip -oq dataset_ready.zip


In [ ]:
!ls /content/dataset


ls: cannot access '/content/dataset': No such file or directory


In [ ]:
!unzip -l dataset_ready.zip | head -20


Archive:  dataset_ready.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2026-03-02 04:11   content/dataset/
        0  2026-03-02 04:11   content/dataset/images/
        0  2026-03-02 04:11   content/dataset/images/val/
   254517  2026-03-02 04:02   content/dataset/images/val/G0032895_e_2_0.jpg
   197350  2026-03-02 04:01   content/dataset/images/val/G0022284_3_4.jpg
   328896  2026-03-02 04:01   content/dataset/images/val/G0032858_2_1.jpg
   306441  2026-03-02 04:01   content/dataset/images/val/G0022241_0_3.jpg
   230852  2026-03-02 04:01   content/dataset/images/val/G0032832_3_0.jpg
   372363  2026-03-02 04:01   content/dataset/images/val/G0022538_1_3.jpg
   322492  2026-03-02 04:00   content/dataset/images/val/G0022066_3_1.jpg
   320149  2026-03-02 04:00   content/dataset/images/val/G0022144_0_1.jpg
   324266  2026-03-02 04:01   content/dataset/images/val/G0022496_1_1.jpg
   238398  2026-03-02 04:02   content/dataset/images/val/P1070_2_0.jpg
   3

In [ ]:
!rm -rf /content/dataset
!rm -rf /content/content


In [ ]:
!unzip -q dataset_ready.zip


In [ ]:
!mv /content/content/dataset /content/
!rm -rf /content/content


In [ ]:
!ls /content/dataset


images	labels	sugarbeet.yaml


In [ ]:
from ultralytics import YOLO


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
model = YOLO('/content/drive/MyDrive/WeedProject/weed_training/exp1/weights/last.pt')


In [ ]:
!cat /content/drive/MyDrive/WeedProject/weed_training/exp1/results.csv | wc -l


7


In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/WeedProject/weed_training/exp1/weights/last.pt')

model.train(
    data='/content/dataset/sugarbeet.yaml',
    epochs=14,   # total for this run
    imgsz=640,
    batch=16,
    project='/content/drive/MyDrive/WeedProject/weed_training',
    name='exp2'
)


Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/sugarbeet.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=14, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/WeedProject/weed_training/exp1/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp2, nbs=64, nms=False, opset=None, optimize=False, 

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!cp /content/drive/MyDrive/WeedProject/dataset_ready.zip /content/


In [ ]:
!unzip -q dataset_ready.zip


In [ ]:
!mv /content/content/dataset /content/
!rm -rf /content/content


In [ ]:
!ls /content/dataset


images	labels	sugarbeet.yaml


In [ ]:
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.2 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/WeedProject/weed_training/exp2/weights/last.pt')

model.train(resume=True)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/sugarbeet.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=14, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7dd498980ad0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q ultralytics


In [ ]:
!cp /content/drive/MyDrive/WeedProject/dataset_ready.zip /content/


In [ ]:
!rm -rf /content/dataset
!unzip -q /content/dataset_ready.zip -d /content/


In [ ]:
import os

if os.path.exists('/content/content/dataset'):
    !mv /content/content/dataset /content/
    !rm -rf /content/content


In [ ]:
!ls /content/dataset


images	labels	sugarbeet.yaml


In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/WeedProject/weed_training/exp2/weights/best.pt')


In [ ]:
model.train(
    data='/content/dataset/sugarbeet.yaml',
    epochs=10,          # continue training
    imgsz=768,
    batch=16,
    cos_lr=True,
    augment=True,
    patience=10,

    # improves 70% metric
    close_mosaic=0,
    degrees=10,
    scale=0.5,

    project='/content/drive/MyDrive/WeedProject/weed_training',
    name='exp3'
)


Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dataset/sugarbeet.yaml, degrees=10, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/WeedProject/weed_training/exp2/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp3, nbs=64, nms=False, opset=None, optimize

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7db099e9df70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/WeedProject/weed_training/exp3/weights/best.pt')

model.train(
    data='/content/dataset/sugarbeet.yaml',
    epochs=5,
    imgsz=768,
    lr0=0.001,
    project='/content/drive/MyDrive/WeedProject/weed_training',
    name='exp4'
)


Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/sugarbeet.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/WeedProject/weed_training/exp3/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp4-2, nbs=64, nms=False, opset=None, op

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7db0eaab7380>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804